# LUNAR Master Pipeline Notebook v3

This notebook is the main audit notebook for the LUNAR repository.

Use it to:

1. Review the pipeline configuration.
2. Run or document the full master pipeline.
3. Confirm that all required workbooks exist.
4. Confirm that all required final figures exist.
5. Inspect key workbooks and generated outputs.

The separate figure notebook is optional and only needed if you want to regenerate or troubleshoot the final figure PNGs.


## 0. Expected final figure filenames

The GitHub figure set should contain these four final PNGs:

```text
Figure_01_Zscore_Heatmaps.png
Figure_02_FullReference_Mahalanobis_Trajectories.png
Figure_03_Bootstrap_Mahalanobis_KDE.png
Figure_04_Bootstrap_Mahalanobis_Violin.png
```


In [ ]:
from pathlib import Path
import json
import pandas as pd

REPO_ROOT = Path(".").resolve()

# Adjust these only if your repository uses different folders.
WORKBOOK_DIRS = [
    REPO_ROOT / "outputs" / "workbooks",
    REPO_ROOT / "outputs",
    REPO_ROOT / "summaries",
]

FIGURE_DIRS = [
    REPO_ROOT / "outputs" / "figures",
    REPO_ROOT / "figures",
    REPO_ROOT / "outputs",
]

print("Repository root:", REPO_ROOT)


## 1. Review configuration

If you are using the master runner, the expected config file is:

```text
lunar_pipeline_config.json
```

If it does not exist yet, copy and edit:

```text
lunar_pipeline_config_TEMPLATE.json
```


In [ ]:
config_candidates = [
    REPO_ROOT / "lunar_pipeline_config.json",
    REPO_ROOT / "pipeline" / "lunar_pipeline_config.json",
    REPO_ROOT / "lunar_pipeline_config_TEMPLATE.json",
]

for p in config_candidates:
    print(("FOUND" if p.exists() else "missing"), p)

config_path = next((p for p in config_candidates if p.exists()), None)

if config_path:
    print("\nUsing:", config_path)
    try:
        cfg = json.loads(config_path.read_text())
        print(json.dumps(cfg, indent=2)[:2500])
    except Exception as exc:
        print("Could not parse config as JSON:", exc)
else:
    print("No config file found.")


## 2. Optional: run the full pipeline

Run this only after confirming that all raw/source files are in the expected locations.

The preferred reproducibility test is the `.py` runner, not the notebook itself.


In [ ]:
# Uncomment this once paths in lunar_pipeline_config.json are correct.

# import subprocess, sys
# subprocess.run([
#     sys.executable,
#     "run_lunar_master_pipeline.py",
#     "--config",
#     "lunar_pipeline_config.json",
# ], check=True)

print("Pipeline run cell is intentionally commented out for safety.")


## 3. Required workbook checklist

This checklist focuses on the workbooks and core datasets that should be present in GitHub.


In [ ]:
required_workbooks = [
    # NHANES
    "LUNAR_NHANES_Data_Quality_Summary_v4.xlsx",
    "LUNAR_NHANES_OVERLAP_Summary_v2.xlsx",

    # Inspiration4
    "Inspiration4_Data_Quality_Summary_v2_SubjectStats.xlsx",
    "LUNAR_Inspiration4_OVERLAP_Summary_v1.xlsx",
    "Inspiration4_Timepoint_Binning_Workbook_All3_Summaries.xlsx",

    # Bed Rest
    "Campaign1_Master_Long_REAL.xlsx",
    "Campaign1_Data_Quality_Summary_REAL.xlsx",
    "LUNAR_BedRest_Campaign1_OVERLAP_Summary_v3.xlsx",
    "BedRest_Timepoint_Binning_Workbook.xlsx",

    # Cross-dataset statistics
    "LUNAR_Final18_Statistical_Comparison_Workbook.xlsx",

    # Mahalanobis
    "LUNAR_Mahalanobis_Full_NHANES_Reference.xlsx",
    "mahalanobis_distance_summary.xlsx",
]

required_datasets = [
    "nhanes_biopro_all_cycles_combined.csv",
    "LUNAR_NHANES_OVERLAP_Data_Wide_v2.csv",
    "Inspiration4_Master_Long.csv",
    "Inspiration4_Master_Wide.csv",
]

def find_file(filename, search_dirs):
    for d in search_dirs:
        candidate = d / filename
        if candidate.exists():
            return candidate
    # Fallback: recursive search from repo root, ignoring .git
    hits = [p for p in REPO_ROOT.rglob(filename) if ".git" not in p.parts]
    return hits[0] if hits else None

workbook_rows = []
for name in required_workbooks:
    found = find_file(name, WORKBOOK_DIRS)
    workbook_rows.append({
        "Type": "Workbook",
        "Required_File": name,
        "Found": found is not None,
        "Path": str(found.relative_to(REPO_ROOT)) if found else "",
    })

dataset_rows = []
for name in required_datasets:
    found = find_file(name, [REPO_ROOT / "data", REPO_ROOT / "outputs", REPO_ROOT / "processed_csv", REPO_ROOT])
    dataset_rows.append({
        "Type": "Dataset",
        "Required_File": name,
        "Found": found is not None,
        "Path": str(found.relative_to(REPO_ROOT)) if found else "",
    })

inventory_df = pd.DataFrame(workbook_rows + dataset_rows)
inventory_df


In [ ]:
missing = inventory_df[~inventory_df["Found"]]
if missing.empty:
    print("✅ All required workbooks/datasets were found.")
else:
    print("⚠️ Missing required files:")
    display(missing)


## 4. Required figure checklist

This is the final PNG figure set for GitHub.


In [ ]:
required_figures = [
    "Figure_01_Zscore_Heatmaps.png",
    "Figure_02_FullReference_Mahalanobis_Trajectories.png",
    "Figure_03_Bootstrap_Mahalanobis_KDE.png",
    "Figure_04_Bootstrap_Mahalanobis_Violin.png",
]

figure_rows = []
for name in required_figures:
    found = find_file(name, FIGURE_DIRS)
    figure_rows.append({
        "Figure": name,
        "Found": found is not None,
        "Path": str(found.relative_to(REPO_ROOT)) if found else "",
    })

figures_df = pd.DataFrame(figure_rows)
figures_df


In [ ]:
missing_figs = figures_df[~figures_df["Found"]]
if missing_figs.empty:
    print("✅ All required figures were found.")
else:
    print("⚠️ Missing figures:")
    display(missing_figs)


## 5. Optional: regenerate final figures

Use this section only if the final PNGs are missing or need to be regenerated.

The updated figure scripts save to the final GitHub filenames:

```text
Figure_01_Zscore_Heatmaps.png
Figure_02_FullReference_Mahalanobis_Trajectories.png
Figure_03_Bootstrap_Mahalanobis_KDE.png
Figure_04_Bootstrap_Mahalanobis_Violin.png
```


In [ ]:
# Example commands. Uncomment as needed after confirming paths.

# import subprocess, sys

# # Figure 01
# subprocess.run([
#     sys.executable,
#     "scripts/cross_dataset/generate_zscore_heatmaps.py",
#     "--input", "outputs/workbooks/LUNAR_Final18_Statistical_Comparison_Workbook.xlsx",
#     "--output", "outputs/figures/Figure_01_Zscore_Heatmaps.png",
# ], check=True)

# # Figure 03
# # Run from the folder containing mahalanobis_distance_summary.xlsx
# subprocess.run([
#     sys.executable,
#     "../../scripts/cross_dataset/generate_realdata_kde_no_phase_titles.py",
# ], cwd="outputs/workbooks", check=True)

print("Figure regeneration commands are provided but commented out.")


## 6. Inspect key workbooks

This section opens each workbook and prints sheet names and dimensions.


In [ ]:
def workbook_summary(path):
    try:
        xl = pd.ExcelFile(path)
        rows = []
        for sheet in xl.sheet_names:
            try:
                df = pd.read_excel(path, sheet_name=sheet, nrows=5)
                # Get full dimensions cheaply where possible.
                full_df = pd.read_excel(path, sheet_name=sheet)
                rows.append({
                    "Workbook": path.name,
                    "Sheet": sheet,
                    "Rows": full_df.shape[0],
                    "Columns": full_df.shape[1],
                })
            except Exception as exc:
                rows.append({
                    "Workbook": path.name,
                    "Sheet": sheet,
                    "Rows": None,
                    "Columns": None,
                    "Error": str(exc)[:120],
                })
        return pd.DataFrame(rows)
    except Exception as exc:
        return pd.DataFrame([{"Workbook": path.name, "Error": str(exc)[:120]}])

summaries = []
for _, row in inventory_df[(inventory_df["Type"] == "Workbook") & (inventory_df["Found"])].iterrows():
    p = REPO_ROOT / row["Path"]
    summaries.append(workbook_summary(p))

if summaries:
    workbook_summary_df = pd.concat(summaries, ignore_index=True)
    display(workbook_summary_df)
else:
    print("No workbooks found to inspect.")


## 7. Inspect cross-dataset statistical workbook

This quickly previews the most important cross-dataset workbook if present.


In [ ]:
stats_path = find_file("LUNAR_Final18_Statistical_Comparison_Workbook.xlsx", WORKBOOK_DIRS)

if stats_path:
    print("Found:", stats_path)
    xl = pd.ExcelFile(stats_path)
    print("Sheets:", xl.sheet_names)

    for sheet in ["NHANES_Reference", "ZScore_Group_Summary", "Cohort_Pairwise_Tests", "I4_RM_ANOVA_Time", "BR_RM_ANOVA_Time"]:
        if sheet in xl.sheet_names:
            df = pd.read_excel(stats_path, sheet_name=sheet)
            print("\n", sheet, df.shape)
            display(df.head())
else:
    print("LUNAR_Final18_Statistical_Comparison_Workbook.xlsx not found.")


## 8. Inspect Mahalanobis workbooks

This checks both the whole-reference and bootstrap-sensitivity Mahalanobis outputs.


In [ ]:
for name in ["LUNAR_Mahalanobis_Full_NHANES_Reference.xlsx", "mahalanobis_distance_summary.xlsx"]:
    path = find_file(name, WORKBOOK_DIRS)
    print("\n===", name, "===")
    if path:
        xl = pd.ExcelFile(path)
        print("Found:", path)
        print("Sheets:", xl.sheet_names)
        for sheet in xl.sheet_names[:4]:
            df = pd.read_excel(path, sheet_name=sheet)
            print(sheet, df.shape)
            display(df.head())
    else:
        print("Not found.")


## 9. Final audit summary

Run this cell last. It reports whether the repository has all final workbooks/datasets and figures.


In [ ]:
all_required_present = inventory_df["Found"].all() and figures_df["Found"].all()

print("Workbook/dataset checklist:")
print(inventory_df["Found"].value_counts(dropna=False).to_string())

print("\nFigure checklist:")
print(figures_df["Found"].value_counts(dropna=False).to_string())

if all_required_present:
    print("\n✅ LUNAR GitHub output inventory is complete.")
else:
    print("\n⚠️ LUNAR GitHub output inventory is incomplete. Review missing files above.")
